In [1]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                all_text += page_text + "\n"
    return all_text

# Extract the text from the PDF
pdf_file_path = './decrypted_output.pdf'
pdf_text = extract_text_from_pdf(pdf_file_path)
print(pdf_text[:1000])  # Print first 1000 characters to check the content


Project Management Institute
PRACTICE STANDARD
FOR PROJECT RISK MANAGEMENT
ISBN: 978-1-933890-38-8
Published by:
Project Management Institute, Inc.
14 Campus Boulevard
Newtown Square, Pennsylvania 19073-3299 USA.
Phone: +610-356-4600
Fax: +610-356-4647
E-mail: customercare@pmi.org
Internet: www.pmi.org
©2009 Project Management Institute, Inc. All rights reserved.
“PMI”, the PMI logo, “PMP”, the PMP logo, “PMBOK”, “PgMP”, “Project Management Journal”, “PM Network”, and the PMI
Today logo are registered marks of Project Management Institute, Inc. The Quarter Globe Design is a trademark of the Project
Management Institute, Inc. For a comprehensive list of PMI marks, contact the PMI Legal Department.
PMI Publications welcomes corrections and comments on its books. Please feel free to send comments on typographical,
formatting, or other errors. Simply make a copy of the relevant page of the book, mark the error, and send it to: Book Editor,
PMI Publications, 14 Campus Boulevard, Newtown Squ

In [2]:
# Save the extracted text to a .txt file
with open('pdf_text_data.txt', 'w', encoding='utf-8') as f:
    f.write(pdf_text)


In [1]:
import tensorflow as tf
from tensorflow import keras
tf.__version__

'2.8.0'

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments

# Load the dataset (your extracted PDF text)
dataset = load_dataset('text', data_files={'train': 'pdf_text_data.txt'})

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Add a new [PAD] token and set it as the padding token
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Tokenize the dataset
def tokenize_function(examples):
    tokens = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=512)
    # GPT-2 requires the labels to be the same as input_ids for causal language modeling
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

# Apply the tokenizer to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Load the pre-trained GPT-2 model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Resize the model's token embeddings to accommodate the new [PAD] token
model.resize_token_embeddings(len(tokenizer))

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for the fine-tuned model
    num_train_epochs=3,              # Number of training epochs
    per_device_train_batch_size=2,   # Reduced batch size to handle RAM usage
    per_device_eval_batch_size=2,    # Reduced evaluation batch size
    gradient_accumulation_steps=4,   # Accumulate gradients over 4 steps to simulate a larger batch
    warmup_steps=500,                # Number of warmup steps
    logging_dir="./logs",            # Directory for logging
    logging_steps=10,
    save_steps=500,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
)

# Start training
trainer.train()


c:\Users\Zaineb\anaconda3\lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  0%|          | 0/1347 [00:00<?, ?it/s]

{'loss': 1.7811, 'grad_norm': 11.777359962463379, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.02}
{'loss': 1.8614, 'grad_norm': 5.673522472381592, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.04}
{'loss': 1.9442, 'grad_norm': 12.368001937866211, 'learning_rate': 3e-06, 'epoch': 0.07}


In [ ]:
# Load the fine-tuned model
fine_tuned_model = GPT2LMHeadModel.from_pretrained('./results')

# Generate text from the fine-tuned model
input_text = "What is the purpose of Quantitative Risk Analysis?"
inputs = tokenizer.encode(input_text, return_tensors='pt')
outputs = fine_tuned_model.generate(inputs, max_length=150)

# Print the generated response
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
